# 01 - Ingestion and PDF Parsing

This notebook reads the source PDF from a Databricks Volume, parses it with `ai_parse_document`,
extracts clean text, and stores the result in a Delta table.

In [0]:
import re
from pyspark.sql import DataFrame

In [0]:
# -----------------------------
# Configuration
# -----------------------------
CATALOG = "workspace"
SCHEMA = "rag_demo"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/rag_volume/"
PDF_FILE_NAME = "Enterprise_Runbook_Library.pdf"
PDF_FILE_PATH = f"{VOLUME_PATH}{PDF_FILE_NAME}"

PARSED_DOCS_TABLE = f"{CATALOG}.{SCHEMA}.parsed_docs_clean"

#-----------------------------
# Print
#-----------------------------
print(VOLUME_PATH)
print(PDF_FILE_PATH)
print(PARSED_DOCS_TABLE)

## Helper Functions

In [0]:
def validate_volume_access(volume_path: str) -> None:
    """Validate access to the Databricks Volume"""
    display(dbutils.fs.ls(volume_path))


def parse_pdf(file_path: str) -> DataFrame:
    """Parse PDF using Databricks AI Functions"""
    return spark.sql(f"""
    SELECT
      '{file_path}' AS file_path,
      ai_parse_document('{file_path}') AS parsed
    """)


def extract_clean_text(parsed_df: DataFrame) -> str:
    """
    Extract content fields from parsed payload and normalize escaped characters
    """
    parsed_string_df = parsed_df.selectExpr(
        "file_path",
        "CAST(parsed AS STRING) AS parsed_str"
    )

    row = parsed_string_df.collect()[0]
    parsed_str = row["parsed_str"]

    contents = re.findall(r'"content":"(.*?)"', parsed_str)

    clean_text = "\n\n".join(contents)
    clean_text = (
        clean_text
        .replace('\\"', '"')
        .replace("\\n", "\n")
        .replace("\\t", " ")
    )

    return clean_text

## Validate source file access

In [0]:
validate_volume_access(VOLUME_PATH)

## Parse the PDF

In [0]:
parsed_df = parse_pdf(PDF_FILE_PATH)
display(parsed_df)

## Extract clean text from parsed output

In [0]:
clean_text = extract_clean_text(parsed_df)
print(clean_text[:3000])

## Save cleaned document to Delta

In [0]:
clean_doc_df = spark.createDataFrame(
    [(PDF_FILE_PATH, PDF_FILE_NAME, clean_text)],
    ["file_path", "file_name", "text"]
)

display(clean_doc_df)
clean_doc_df.write.mode("overwrite").saveAsTable(PARSED_DOCS_TABLE)